# 03. Preprocessing and PCA

This notebook constructs the shared modeling representation. The current ChemDFM pipeline fits PCA on the training subset only, then transforms test and OOD cells with the same projection. That choice is preserved here because it avoids information leakage while providing a compact latent space for residual prediction. The notebook also creates label encoders and control anchors in both PCA space and gene space, thereby bridging model training and biological evaluation.


In [1]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 108.1 MB/s eta 0:00:00
Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM_repo
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [4]:
from pathlib import Path
import json, os, random, pickle, warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

for p in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR, RUNS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULT_H5AD = RAW_DIR / "sciplex_complete_middle_subset.h5ad"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_H5AD =", DEFAULT_H5AD)


import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, average_precision_score
from scipy.stats import pearsonr, spearmanr

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Using device:", DEVICE)


PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DEFAULT_H5AD = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [5]:
import anndata as ad

DATA_PATH = DEFAULT_H5AD
assert DATA_PATH.exists(), f"Missing data file: {DATA_PATH}"

adata = ad.read_h5ad(DATA_PATH)
print(adata)
print("obs columns:", list(adata.obs.columns))

if "dose_val" in adata.obs.columns and "dose" not in adata.obs.columns:
    adata.obs["dose"] = adata.obs["dose_val"]

X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)


AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
obs columns: ['cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_sco

In [6]:
SPLIT_COL = "split_ho_pathway"
adata.obs["_split3"] = adata.obs[SPLIT_COL].astype(str).map(map_split)
keep_mask = adata.obs["_split3"].isin(["train", "test", "ood"]).values
adata = adata[keep_mask].copy()
X = X[keep_mask]

train_mask = (adata.obs["_split3"] == "train").values
print(adata.obs["_split3"].value_counts())


_split3
train    292283
test      51798
ood       10559
Name: count, dtype: int64


## Train-only PCA fit

The PCA model is estimated using only the training subset and then applied to test and OOD cells. The transformed matrices and the fitted PCA object are saved explicitly because all downstream notebooks should reuse exactly the same latent representation.


In [7]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

PCA_DIM = 25
pca = PCA(n_components=PCA_DIM, random_state=SEED)
X_pca = np.zeros((adata.n_obs, PCA_DIM), dtype=np.float32)
X_pca[train_mask] = pca.fit_transform(X[train_mask]).astype(np.float32)
X_pca[~train_mask] = pca.transform(X[~train_mask]).astype(np.float32)

(PROCESSED_DIR / "pca").mkdir(parents=True, exist_ok=True)
np.save(PROCESSED_DIR / "pca" / f"X_pca_{PCA_DIM}d.npy", X_pca)
with open(PROCESSED_DIR / "pca" / f"pca_model_{PCA_DIM}d.pkl", "wb") as f:
    pickle.dump(pca, f)

pca_report = pd.DataFrame({
    "component": np.arange(1, PCA_DIM + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_)
})
pca_report.to_csv(PROCESSED_DIR / "pca" / "pca_variance_report.csv", index=False)
pca_report.tail()


,component,explained_variance_ratio,cumulative_explained_variance
20,21,0.002860,0.276295
21,22,0.002839,0.279135
22,23,0.002808,0.281942
23,24,0.002711,0.284653
24,25,0.002687,0.287340


## Encoders and control anchors

The response model requires stable integer encodings for condition and cell type. In addition, residual targets are defined relative to the mean control profile of each cell type. The notebook therefore saves both encoders and both types of control anchors.


In [9]:
drug_enc = LabelEncoder()
cell_enc = LabelEncoder()
drug_enc.fit(adata.obs["condition"].astype(str))
cell_enc.fit(adata.obs["cell_type"].astype(str))

adata.obs["drug_idx"] = drug_enc.transform(adata.obs["condition"].astype(str))
adata.obs["cell_idx"] = cell_enc.transform(adata.obs["cell_type"].astype(str))

(PROCESSED_DIR / "encoders").mkdir(parents=True, exist_ok=True)
with open(PROCESSED_DIR / "encoders" / "drug_encoder.pkl", "wb") as f:
    pickle.dump(drug_enc, f)
with open(PROCESSED_DIR / "encoders" / "cell_encoder.pkl", "wb") as f:
    pickle.dump(cell_enc, f)

control_mask = (adata.obs["condition"].astype(str) == "control").values
ctrl_means_pca = {}
ctrl_means_gene = {}
ctrl_counts = {}
for cell in sorted(adata.obs["cell_type"].astype(str).unique()):
    m = control_mask & (adata.obs["cell_type"].astype(str).values == cell)
    if m.sum() == 0:
        raise ValueError(f"No control cells for {cell}")
    ctrl_means_pca[cell] = X_pca[m].mean(axis=0).astype(np.float32)
    ctrl_means_gene[cell] = X[m].mean(axis=0).astype(np.float32)
    ctrl_counts[cell] = int(m.sum())

(PROCESSED_DIR / "controls").mkdir(parents=True, exist_ok=True)
pd.DataFrame(ctrl_means_pca).T.to_csv(PROCESSED_DIR / "controls" / f"control_mean_pca_{PCA_DIM}d.csv")
pd.DataFrame(ctrl_means_gene).T.to_csv(PROCESSED_DIR / "controls" / "control_mean_gene.csv")
save_json(ctrl_counts, PROCESSED_DIR / "controls" / "control_counts_by_cell_type.json")
(PROCESSED_DIR / "datasets").mkdir(parents=True, exist_ok=True)
X0_pca = np.stack([ctrl_means_pca[c] for c in adata.obs["cell_type"].astype(str).values]).astype(np.float32)
DELTA_pca = (X_pca - X0_pca).astype(np.float32)

np.save(PROCESSED_DIR / "datasets" / f"X0_pca_{PCA_DIM}d.npy", X0_pca)
np.save(PROCESSED_DIR / "datasets" / f"DELTA_pca_{PCA_DIM}d.npy", DELTA_pca)
adata.obs.to_parquet(PROCESSED_DIR / "datasets" / "adata_obs_processed.parquet", index=False)

print("Saved processed artifacts.")


Saved processed artifacts.


In [10]:
(PROCESSED_DIR / 'datasets').mkdir(parents=True, exist_ok=True)


The saved latent matrix, encoders, and control anchors define the canonical representation used throughout the rest of the study. The next notebook establishes a strong performance floor by evaluating trivial and structured baselines under the same split and metric definitions.
